# 31 · Join a table to the study area against a pandas merge

**Question.** Snow's Table VIII gives 32 sub-districts a population and a death count. The study
area carries **33** named parts, because Streatham is two polygons. Does each part end up with the
rate belonging to its own name — and does the join still refuse a name it should not accept?

This is the step that turns a table of rates into a map of rates. It is the choropleth the South
London example argues for in prose, and it shipped on 2026-10-25 with no independent evidence.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "rapidfuzz", "jellyfish"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## What is independently checkable, and what is not

**Not the normalisation.** `normaliseName` is Fieldwork's own. A check that reimplemented it would
be this author's arithmetic checking this author's arithmetic — the trap check 27's record names
and refuses. So three separable things are recomputed, and the check says which is which:

| | what it is |
| --- | --- |
| **The merge, given keys** | fully independent. `pandas.merge`, left on the parts, so every part survives |
| **The raw-string baseline** | independent in the strongest sense available: equality needs no normalisation at all |
| **What the folding is *allowed* to do** | a property, not a reimplementation. Every folded key must equal its name with all non-alphanumerics deleted, and no two distinct names may fold together |

That third one is weaker than an independent fold, and it is not nothing: it refuses a folding
that expanded `St.` to `Saint`, dropped a word, folded two names together, or touched a letter.


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_31_area_join import run

report = run(Path.cwd() / "fixtures" / "area-join-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['note']}   [{c['normalisation']}]")
    print(f"  parts matched   Fieldwork {c['fieldwork']['matchedParts']:>2}   pandas {c['pandas']['matchedParts']:>2}")
    print(f"  rows matched              {c['fieldwork']['matchedRows']:>2}          {c['pandas']['matchedRows']:>2}")
    print(f"  shared parts              {c['fieldwork']['sharedParts']:>2}          {c['pandas']['sharedParts']:>2}")
    if c["unmatchedPartNames"]:
        print(f"  unmatched: {', '.join(c['unmatchedPartNames'])}")
    print(f"  diagnostics: {', '.join(c['diagnosticCodes'])}")
    for name, ok in c["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")


## What the folding bought

The raw-string baseline is the measurement that makes the folding worth anything. Nine of the 32
rows match only once punctuation is folded — the table writes `St. George, Camberwell` where the
boundary file writes `St. George  Camberwell`, with a doubled space and no comma. A join on raw
keys would have drawn a choropleth missing **28% of its sub-districts**, every one of them a name
with a comma in it.

The cell below lists each pair and confirms the two differ *only* in characters that are not
letters or digits — the property, checked rather than assumed.


In [ ]:
g = report["foldingGain"]
print(f"raw {g['rawMatchedRows']} of 32 rows  ->  folded {g['foldedMatchedRows']} of 32"
      f"   ({g['rowsTheFoldingWon']} rows, {g['shareOfRows']:.0%} of the table)\n")
for p in g["pairs"]:
    mark = "OK " if p["differsOnlyInPunctuation"] else "DIFF"
    print(f"  {mark} boundary {p['boundary']!r}")
    print(f"       table {p['row']!r}")
print(f"\n{g['note']}")

f = report["folding"]
print(f"\nfolding touched nothing else: {f['agrees']}  ({f['partsChecked']} parts)")
print("criterion:", f["criterion"])
for problem in f["problems"]:
    print("  PROBLEM", problem)


## The negative control, and what the refusal is worth

Snow's table and these boundaries have **no pair that differs by more than punctuation**. So
nothing in the example tests a name that *ought* not to match, and 32 of 32 is also the score of a
rule that matched too readily. The fixture therefore renames one part to `Camberwell Green` — a
real place the table does not carry — and the join must report it rather than find it a home.

`rapidfuzz` and `jellyfish` are carried on this case alone, in the role the score interval plays
in check 27: **never asserted, only reported.** They say how close `Camberwell Green` is to
`Camberwell`, which is how close a similarity-based linkage would have come to accepting it. This
widget is a keyed join with no threshold and no review band, and that number is what the decision
is worth.


In [ ]:
n = report["negativeControl"]
print("renamed    :", f"{n['renamedFrom']!r} -> {n['renamedTo']!r}")
print("constructed:", n["constructed"])
print("unmatched  :", n["unmatched"])
print()
for name, value in n["similarityToTheNameItReplaced"].items():
    print(f"  {name:24} {value}")
print()
print(n["whatTheRefusalIsWorth"])
print()
for name, ok in n["checks"].items():
    print(f"  {'OK ' if ok else 'DIFF'} {name}")


## What this cannot establish

**That folding punctuation is the right rule for names.** It is right for these two files, where
every difference is a comma or a doubled space. A boundary spelling `St Mary` against a table's
`Saint Mary` would fail this join — and whether failing is right *there* is a judgement about the
data, not an outcome of this check. The practitioner declares a synonym or fixes the source.

**That the folding itself is correctly implemented.** Not reimplemented here, on purpose: a second
copy written by the same author is not a second opinion.

**That the 33 parts are the right geography.** Streatham being two polygons is a fact about the
digitisation. Both parts take the same rate, which is correct for a rate and would be wrong for a
count.

**That a sub-district rate measures an exposure.** In the middle 16 sub-districts both companies
supplied the same streets, so the rate there mixes two exposures — Snow's own confounding, and why
he went house to house.


In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
b = report["fixture"]["boundaryProvenance"]
print("boundary :", b["source"])
print("licence  :", b["licence"], "·", b["licenceSource"])
print("fixture  :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas   :", report["environment"]["pandas"])
print("runtime  :", report["environment"]["runtime"])
